In [ ]:
from langgraph.graph import StateGraph
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
model = ChatOpenAI()

In [ ]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [ ]:
def create_outline(state: blogState) -> BlogState:
    # fetch title and outline
    title = state['title']
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'

    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline
    return state

In [ ]:
# Create blog
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f'write a detailed blog on the title 0 {title} using the following outline \n {outline}'
    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [ ]:
# create the graph
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outlines', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [ ]:
inital_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(inital_state)
print(final_state)